# Painel — efeitos municipais e temporais

Seis ajustes na mesma amostra por destino. Preços anuais comuns não entram isoladamente no TWFE. Aptidão e biomas invariantes também são absorvidos. Interações retêm variação residual (auditoria). SE agrupados em município e ano; inferência t pelo menor número de clusters menos um.

**Inferência:** se a matriz two-way tiver autovalor negativo, aplica-se projeção espectral (autovalores negativos iguais a zero). SE bruto permanece exportado para auditoria; a correção é determinada pelo diagnóstico, nunca por significância. Sensibilidade municipal/HAC também é salva. A projeção não resolve poucos clusters nem ausência de identificação.

In [1]:
from utilitarios import *
from IPython.display import display
d = ler()
print(f"{len(d)} linhas, {d.id.nunique()} municípios, {d.ano.min()}–{d.ano.max()}")

3384 linhas, 141 municípios, 2001–2024


## 1. Preços dentro dos municípios — dois destinos

In [2]:
for uso in ['agricultura','pastagem']:
    ident='P01_'+uso; y='pp_'+uso
    s=amostra(d,[y,'z_p_soja','z_p_milho','z_p_boi','expo_soja','expo_milho','fiscal','preco_fiscal'],ident)
    formula=f'{y} ~ z_p_soja + z_p_milho + z_p_boi + fiscal + EntityEffects'
    r,cov,cm=estimar(formula,s,'PainelOLS')
    display(exportar(ident,'Preços dentro dos municípios',formula,s,r,cov,'PainelOLS','municipio',y,alerta='fiscalização endógena; GAEZ incorpora terras de 2010; interações não são retorno líquido'))
    # Sensibilidade de SE, sem novo ajuste de especificação.
    rm=PanelOLS.from_formula(formula,s.set_index(['id','ano']),drop_absorbed=False).fit(cov_type='clustered',cluster_entity=True)
    pd.DataFrame({'id':ident,'termo':r.params.index,'se_cluster_municipio':rm.std_errors.values}).to_csv(PASTA/f'sensibilidade_{ident}.csv',index=False)

,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,P01_agricultura,z_p_soja,-0.078736,0.080501,-0.245686,0.088213,0.338670,1140,fiscalização endógena; GAEZ incorpora terras d...
1,P01_agricultura,z_p_milho,0.065722,0.068222,-0.075761,0.207205,0.345838,1140,fiscalização endógena; GAEZ incorpora terras d...
2,P01_agricultura,z_p_boi,-0.037535,0.030530,-0.100851,0.025780,0.231886,1140,fiscalização endógena; GAEZ incorpora terras d...
3,P01_agricultura,fiscal,0.000565,0.003979,-0.007688,0.008817,0.888448,1140,fiscalização endógena; GAEZ incorpora terras d...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,P01_pastagem,z_p_soja,-0.245622,0.237808,-0.738805,0.247561,0.312890,1140,fiscalização endógena; GAEZ incorpora terras d...
1,P01_pastagem,z_p_milho,0.194815,0.217356,-0.255954,0.645584,0.379798,1140,fiscalização endógena; GAEZ incorpora terras d...
2,P01_pastagem,z_p_boi,-0.068634,0.095491,-0.266670,0.129402,0.479861,1140,fiscalização endógena; GAEZ incorpora terras d...
3,P01_pastagem,fiscal,-0.033701,0.020310,-0.075821,0.008420,0.111245,1140,fiscalização endógena; GAEZ incorpora terras d...


## 2. Resposta diferencial conforme aptidão — dois destinos

In [3]:
for uso in ['agricultura','pastagem']:
    ident='P02_'+uso; y='pp_'+uso
    s=amostra(d,[y,'z_p_soja','z_p_milho','z_p_boi','expo_soja','expo_milho','fiscal','preco_fiscal'],ident)
    formula=f'{y} ~ expo_soja + expo_milho + fiscal + EntityEffects + TimeEffects'
    r,cov,cm=estimar(formula,s,'PainelOLS')
    display(exportar(ident,'Resposta diferencial conforme aptidão',formula,s,r,cov,'PainelOLS','municipio+ano',y,alerta='fiscalização endógena; GAEZ incorpora terras de 2010; interações não são retorno líquido'))
    # Sensibilidade de SE, sem novo ajuste de especificação.
    rm=PanelOLS.from_formula(formula,s.set_index(['id','ano']),drop_absorbed=False).fit(cov_type='clustered',cluster_entity=True)
    pd.DataFrame({'id':ident,'termo':r.params.index,'se_cluster_municipio':rm.std_errors.values}).to_csv(PASTA/f'sensibilidade_{ident}.csv',index=False)

,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,P02_agricultura,expo_soja,0.007733,0.007818,-0.008481,0.023946,0.333376,1140,fiscalização endógena; GAEZ incorpora terras d...
1,P02_agricultura,expo_milho,0.002689,0.002890,-0.003304,0.008681,0.362254,1140,fiscalização endógena; GAEZ incorpora terras d...
2,P02_agricultura,fiscal,0.006249,0.002293,0.001493,0.011005,0.012363,1140,fiscalização endógena; GAEZ incorpora terras d...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,P02_pastagem,expo_soja,0.068627,0.032972,0.000247,0.137008,0.049246,1140,fiscalização endógena; GAEZ incorpora terras d...
1,P02_pastagem,expo_milho,0.057130,0.040734,-0.027347,0.141606,0.174717,1140,fiscalização endógena; GAEZ incorpora terras d...
2,P02_pastagem,fiscal,0.008698,0.011616,-0.015392,0.032789,0.461895,1140,fiscalização endógena; GAEZ incorpora terras d...


## 3. Modulação associativa pela fiscalização — dois destinos

In [4]:
for uso in ['agricultura','pastagem']:
    ident='P03_'+uso; y='pp_'+uso
    s=amostra(d,[y,'z_p_soja','z_p_milho','z_p_boi','expo_soja','expo_milho','fiscal','preco_fiscal'],ident)
    formula=f'{y} ~ expo_soja + expo_milho + fiscal + preco_fiscal + EntityEffects + TimeEffects'
    r,cov,cm=estimar(formula,s,'PainelOLS')
    display(exportar(ident,'Modulação associativa pela fiscalização',formula,s,r,cov,'PainelOLS','municipio+ano',y,alerta='fiscalização endógena; GAEZ incorpora terras de 2010; interações não são retorno líquido'))
    # Sensibilidade de SE, sem novo ajuste de especificação.
    rm=PanelOLS.from_formula(formula,s.set_index(['id','ano']),drop_absorbed=False).fit(cov_type='clustered',cluster_entity=True)
    pd.DataFrame({'id':ident,'termo':r.params.index,'se_cluster_municipio':rm.std_errors.values}).to_csv(PASTA/f'sensibilidade_{ident}.csv',index=False)

,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,P03_agricultura,expo_soja,0.009868,0.009266,-0.009347,0.029084,0.298394,1140,fiscalização endógena; GAEZ incorpora terras d...
1,P03_agricultura,expo_milho,0.002350,0.002674,-0.003196,0.007896,0.388965,1140,fiscalização endógena; GAEZ incorpora terras d...
2,P03_agricultura,fiscal,0.006181,0.002280,0.001453,0.010908,0.012758,1140,fiscalização endógena; GAEZ incorpora terras d...
3,P03_agricultura,preco_fiscal,0.003484,0.004187,-0.005200,0.012168,0.414340,1140,fiscalização endógena; GAEZ incorpora terras d...


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,P03_pastagem,expo_soja,0.051920,0.037641,-0.026142,0.129983,0.181646,1140,fiscalização endógena; GAEZ incorpora terras d...
1,P03_pastagem,expo_milho,0.059775,0.040395,-0.023998,0.143548,0.153112,1140,fiscalização endógena; GAEZ incorpora terras d...
2,P03_pastagem,fiscal,0.009237,0.011182,-0.013953,0.032426,0.417648,1140,fiscalização endógena; GAEZ incorpora terras d...
3,P03_pastagem,preco_fiscal,-0.027253,0.019753,-0.068219,0.013713,0.181553,1140,fiscalização endógena; GAEZ incorpora terras d...
